In [1]:
import pandas as pd
import numpy as np 

betaCaucasusCA = pd.read_csv('../Data/Processed/weeklyBetaCaucasusCA.csv')
betaMiddleEast = pd.read_csv('../Data/Processed/weeklyBetaMiddleEast.csv')
betaNorthAfrica = pd.read_csv('../Data/Processed/weeklyBetaNorthAfrica.csv')
betaEurope = pd.read_csv('../Data/Processed/weeklyBetaEurope.csv')

betaAll = pd.concat([betaCaucasusCA, betaMiddleEast, betaNorthAfrica, betaEurope])

print(betaAll.shape)
print(betaAll['region'].value_counts())

(26979, 5)
region
Europe                       16031
Middle East                   5083
Caucasus and Central Asia     3519
Northern Africa               2346
Name: count, dtype: int64


In [4]:
def buildBetaReport(group): 
    sortedBetas = group.sort_values('betaMean', ascending=False).reset_index(drop=True)

    meanAbsBeta = group['betaMean'].abs().mean()
    maxBeta = sortedBetas.loc[0, 'betaMean']
    maxBetaStd = sortedBetas.loc[0, 'betaStd']
    topCountry = sortedBetas.loc[0, 'country']

    if len(sortedBetas) > 1: 
        secondBeta = sortedBetas.loc[1, 'betaMean']
        secondCountry = sortedBetas.loc[1, 'country']
    else: 
        secondBeta = np.nan
        secondCountry = None

    topGap = maxBeta - secondBeta

    return pd.Series({
        'meanAbsBeta' : meanAbsBeta, 
        'maxBeta' : maxBeta, 
        'maxStd' : maxBetaStd,
        'topCountry' : topCountry, 
        'secondCountry' : secondCountry, 
        'firstMinusSecond' : topGap
    })

betaFeatures = betaAll.groupby(['week', 'region']).apply(buildBetaReport, include_groups=False).reset_index()

print(betaFeatures.shape)
betaFeatures.head()

(1564, 8)


,week,region,meanAbsBeta,maxBeta,maxStd,topCountry,secondCountry,firstMinusSecond
0,2018-01-01/2018-01-07,Caucasus and Central Asia,1.242547,2.221638,0.163267,Afghanistan,Azerbaijan,0.033687
1,2018-01-01/2018-01-07,Europe,0.275416,2.474768,0.234720,Ukraine,Bulgaria,1.568269
2,2018-01-01/2018-01-07,Middle East,1.048720,2.471036,0.078271,Syria,Yemen,1.261022
3,2018-01-01/2018-01-07,Northern Africa,0.540358,0.804794,0.184471,Sudan,Egypt,0.516772
4,2018-01-08/2018-01-14,Caucasus and Central Asia,1.485593,2.404007,0.127639,Afghanistan,Azerbaijan,0.122699


In [7]:
betaFeatures = betaFeatures.sort_values(['region', 'week']).reset_index(drop=True)

betaFeatures['meanAbsBetaChange'] = betaFeatures.groupby('region')['meanAbsBeta'].diff()
betaFeatures['maxBetaChange'] = betaFeatures.groupby('region')['maxBeta'].diff()

print(betaFeatures.shape)
betaFeatures.head()

(1564, 10)


,week,region,meanAbsBeta,maxBeta,maxStd,topCountry,secondCountry,firstMinusSecond,meanAbsBetaChange,maxBetaChange
0,2018-01-01/2018-01-07,Caucasus and Central Asia,1.242547,2.221638,0.163267,Afghanistan,Azerbaijan,0.033687,NaN,NaN
1,2018-01-08/2018-01-14,Caucasus and Central Asia,1.485593,2.404007,0.127639,Afghanistan,Azerbaijan,0.122699,0.243047,0.182370
2,2018-01-15/2018-01-21,Caucasus and Central Asia,1.477888,2.449681,0.160385,Afghanistan,Azerbaijan,0.392742,-0.007705,0.045674
3,2018-01-22/2018-01-28,Caucasus and Central Asia,1.484648,2.350285,0.136129,Afghanistan,Azerbaijan,0.112825,0.006760,-0.099396
4,2018-01-29/2018-02-04,Caucasus and Central Asia,1.366916,2.068428,0.122894,Azerbaijan,Afghanistan,0.050741,-0.117732,-0.281858


In [8]:
weeklyComposite = pd.read_csv('../Data/Processed/weeklyCompositeUncertainty.csv')
weeklyComposite['week'] = pd.PeriodIndex(weeklyComposite['week'], freq='W-SUN')

betaFeatures['week'] = pd.PeriodIndex(betaFeatures['week'], freq='W-SUN')

featureTable = betaFeatures.merge(weeklyComposite, on=['week', 'region'], how='inner')

print(featureTable.shape)
featureTable.columns.tolist()

(1564, 14)


['week',
 'region',
 'meanAbsBeta',
 'maxBeta',
 'maxStd',
 'topCountry',
 'secondCountry',
 'firstMinusSecond',
 'meanAbsBetaChange',
 'maxBetaChange',
 'meanUncertainty',
 'hdiOverlapRate',
 'entropy',
 'entropyReliable']

In [9]:
print(featureTable.isna().sum())

week                 0
region               0
meanAbsBeta          0
maxBeta              0
maxStd               0
topCountry           0
secondCountry        0
firstMinusSecond     0
meanAbsBetaChange    4
maxBetaChange        4
meanUncertainty      0
hdiOverlapRate       0
entropy              0
entropyReliable      0
dtype: int64


In [10]:
featureTable = featureTable.dropna(subset=['meanAbsBetaChange', 'maxBetaChange']).reset_index(drop=True)
print(featureTable.shape)

(1560, 14)


In [11]:
featureTable.to_csv('../Data/Processed/featureTable.csv', index=False)